In [1]:


pip install openai-agents openai pandas python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [2]:
%reset -f

import os
import json
from dotenv import load_dotenv
from openai import AsyncOpenAI
from pydantic import BaseModel
from IPython.display import display, HTML
from agents import Agent, ModelSettings, Runner, function_tool, set_tracing_disabled
from agents.models.openai_chatcompletions import OpenAIChatCompletionsModel

load_dotenv()
os.makedirs("output", exist_ok=True)

# Same setup pattern as the demos: explicit model object + settings
set_tracing_disabled(True)

if os.environ.get("OPENROUTER_API_KEY"):
    client = AsyncOpenAI(
        api_key=os.environ["OPENROUTER_API_KEY"],
        base_url="https://openrouter.ai/api/v1",
    )
    AGENTS_MODEL = OpenAIChatCompletionsModel(model="openai/gpt-4o-mini", openai_client=client)
elif os.environ.get("OPENAI_API_KEY"):
    client = AsyncOpenAI(api_key=os.environ["OPENAI_API_KEY"])
    AGENTS_MODEL = OpenAIChatCompletionsModel(model="gpt-4o-mini", openai_client=client)
else:
    raise ValueError("Set OPENROUTER_API_KEY or OPENAI_API_KEY in .env")

SETTINGS = ModelSettings(temperature=0, max_tokens=4096)

print("Setup complete")

Setup complete


Load data

In [3]:
import os
print(os.getcwd())

/Users/carriesu/Documents/final-project-jiaxin-su1/notebooks


In [4]:
import pandas as pd
import numpy as np
%pip install rank_bm25
from rank_bm25 import BM25Okapi
import re
# mapping course prefix -> full dept name
dept_dict = {
    "ANTH": "anthropology",
    "ART": "art",
    "ARTHI": "art history",
    "AS AM": "asian american study",
    "BIOE": "biological engineering",
    "BL ST": "black studies",
    "CH E": "chemical engineering",
    "CHEM": "chemistry",
    "CH ST": "chicana and chicano studies",
    "CHIN": "chinese",
    "CLASS": "classics",
    "COMM": "communication",
    "C LIT": "comparative literature",
    "CMPSC": "computer science",
    "CNCSP": "counseling clinical school psychology",
    "DANCE": "dance",
    "DYNS": "dynamical neuroscience",
    "EARTH": "earth science",
    "EACS": "east asian languages cultural studies",
    "EEMB": "ecology evolution and marine biology",
    "ECON": "economics",
    "ED": "education",
    "ECE": "electrical computer engineering",
    "ENGR": "engineering",
    "ENGL": "english",
    "ESM": "environmental science and management",
    "ENV S": "environmental studies",
    "ESS": "exercise and sport studies",
    "FEMST": "feminist",
    "FAMST": "film and media studies",
    "FR": "french",
    "GEOG": "geography",
    "GER": "german",
    "GLOBL": "global studies",
    "GREEK": "greek",
    "HEB": "hebrew",
    "HIST": "history",
    "IQB": "quantitative biosciences",
    "INT": "interdisciplinary",
    "ITAL": "italian",
    "JAPAN": "japanese",
    "KOR": "korean",
    "LATIN": "latin",
    "LAIS": "latin american and iberian studies",
    "LING": "linguistics",
    "MARSC": "marine science",
    "MATRL": "materials",
    "MATH": "mathematics",
    "ME": "mechanical engineering",
    "MAT": "media arts and technology",
    "ME ST": "mechanical engineering",
    "MS": "army physical training",
    "MCDB": "molecular cellular and developmental biology",
    "MUS": "music",
    "MUS A": "music performance audition",
    "PHIL": "philosophy",
    "PHYS": "physics",
    "PORT": "portuguese",
    "PSY": "psychology",
    "RENST": "renewable and sustainable technology",
    "RUSS": "russian",
    "SLAV": "german and slavic studies",
    "SOC": "sociology",
    "SPAN": "spanish",
    "PSTAT": "statistics applied probability data science",
    "THTR": "theater and dance",
    "WRIT": "writing",
}

def parse_dept(course_id: str) -> str:
    """Map course prefix to a full department name."""
    if not isinstance(course_id, str):
        return ""
    cid = course_id.strip().upper()
    # Try longest matches first
    sorted_keys = sorted(dept_dict.keys(), key=len, reverse=True)
    for k in sorted_keys:
        if cid.startswith(k.upper()):
            return dept_dict[k]
    return ""

def remove_unwanted_words(text: str) -> str:
    """
    Cleans text by:
    - lowercasing
    - removing punctuation except spaces
    - removing certain unwanted words
    """
    UNWANTED_WORDS = {"group", "club", "team", "organization"}
    if not isinstance(text, str):
        text = str(text)
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]+', '', text)
    tokens = text.split()
    filtered_tokens = [t for t in tokens if t not in UNWANTED_WORDS]
    return " ".join(filtered_tokens)

def get_course_number(course_id: str):
    """
    Extract the numeric part at the end of a course ID.
    Examples:
        'PHYS 99' -> 99
        'CNCSP 199' -> 199
        'PSY 108' -> 108
    Returns None if no number is found.
    """
    if not isinstance(course_id, str):
        return None
    match = re.search(r'(\d{1,3})\s*$', course_id.strip())
    if match:
        return int(match.group(1))
    return None

# Read in courses CSV
df_courses = pd.read_csv("../data/Fall2024_courses.csv")

# remove courses with course numbers 99 and 199, which are typically special topics courses
df_courses_filtered = df_courses.copy()
df_courses_filtered["course_number"] = df_courses_filtered["courseId"].apply(get_course_number)

df_courses_filtered = df_courses_filtered[
    ~df_courses_filtered["course_number"].isin([99, 199])
].copy()

# Create a 'merged_text' that includes title + description
df_courses_filtered["merged_text"] = (
    df_courses_filtered["title"].fillna("").astype(str) + " " +
    df_courses_filtered["description"].fillna("").astype(str)
)

# Add department name to the merged text
df_courses_filtered["dept_fullname"] = df_courses_filtered["courseId"].apply(parse_dept)
df_courses_filtered["merged_text"] = (
    df_courses_filtered["merged_text"] + " " + df_courses_filtered["dept_fullname"].fillna("")
)

# Clean the final merged_text
df_courses_filtered["merged_text_clean"] = df_courses_filtered["merged_text"].apply(remove_unwanted_words)

tokenized_courses = [doc.split() for doc in df_courses_filtered["merged_text_clean"]]

bm25 = BM25Okapi(tokenized_courses)

def parse_last_three_digits(course_id: str) -> int:
    """
    Returns the integer value of the last 1–3 digits of courseId.
    If no digits are found, returns 9999 (meaning 'no match').
    """
    pattern = r'(\d{1,3})$'  # captures up to 3 digits at the end
    match = re.search(pattern, course_id)
    if match:
        return int(match.group(1))
    return 9999

Note: you may need to restart the kernel to use updated packages.


In [5]:
import re

# Assuming df_students is already loaded, e.g.:
df_students = pd.read_csv("../data/student_data.csv")

# 1) Define a simple text-cleaning helper (inline for convenience)
def remove_unwanted_words(text: str) -> str:
    """
    Cleans text by:
    - lowercasing
    - removing punctuation except spaces
    - removing specific unwanted words (example set below)
    """
    UNWANTED_WORDS = {"group", "club", "team", "organization"}
    if not isinstance(text, str):
        text = str(text)
    text = text.lower()
    # remove non-alphanumeric characters except spaces
    text = re.sub(r'[^a-z0-9\s]+', '', text)
    # split into tokens
    tokens = text.split()
    # remove certain unwanted words
    filtered_tokens = [t for t in tokens if t not in UNWANTED_WORDS]
    return " ".join(filtered_tokens)

# 2) Create new columns for weighted fields:
df_students['tokens_academic_interest'] = None
df_students['tokens_major'] = None
df_students['tokens_research_interest'] = None
df_students['tokens_unweighted'] = None
df_students['year_of_study_cleaned'] = ""


# 5) Iterate through rows and process each
for i, row in df_students.iterrows():
    # --- Weighted fields ---
    academic_text  = remove_unwanted_words(str(row.get('AcademicInterest', '')))
    major_text     = remove_unwanted_words(str(row.get('Major', '')))
    research_text  = remove_unwanted_words(str(row.get('ResearchInterests', '')))

    # Assign tokens back to the DataFrame
    df_students.at[i, 'tokens_academic_interest'] = academic_text.split()
    df_students.at[i, 'tokens_major']            = major_text.split()
    df_students.at[i, 'tokens_research_interest'] = research_text.split()

    # --- Unweighted fields ---
    extra_text  = remove_unwanted_words(str(row.get('ExtracurricularActivities', '')))
    skills_text = remove_unwanted_words(str(row.get('Skills', '')))
    clubs_text  = remove_unwanted_words(str(row.get('ClubMemberships', '')))
    langs_text  = remove_unwanted_words(str(row.get('Languages', '')))

    # Merge these unweighted fields into one string
    combined_unweighted = (
        extra_text + " "
        + skills_text + " "
        + clubs_text + " "
        + langs_text
    ).strip()

    df_students.at[i, 'tokens_unweighted'] = combined_unweighted.split()

    # --- Clean YearOfStudy ---
    year_of_study = str(row.get('YearOfStudy', '')).strip().lower()
    df_students.at[i, 'year_of_study_cleaned'] = year_of_study

# 6) Now df_students has these new columns in place.
print(df_students.head())


   StudentID       Name  AcademicInterest ExtracurricularActivities  \
0          1  Student 1        Psychology               Debate Club   
1          2  Student 2        Psychology               Debate Club   
2          3  Student 3           History           Volunteer Group   
3          4  Student 4  Computer Science           Volunteer Group   
4          5  Student 5  Computer Science               Sports Team   

                                              Skills  Location YearOfStudy  \
0                                    Problem Solving  New York    Freshman   
1  Leadership, Problem Solving, Public Speaking, ...    Boston    Graduate   
2  Data Analysis, Leadership, Public Speaking, Ar...   Chicago      Junior   
3    Public Speaking, Data Analysis, Problem Solving   Chicago    Graduate   
4                                      Data Analysis   Chicago    Graduate   

              Major   GPA                                    Languages  \
0        Psychology  3.27   Ch

In [6]:
from rank_bm25 import BM25Okapi

def parse_last_three_digits(course_id: str) -> int:
    """Extract integer from the last 1–3 digits of course_id. To recommend course based on year of study"""
    pattern = r'(\d{1,3})$'
    match = re.search(pattern, course_id)
    if match:
        return int(match.group(1))
    return 9999

def create_bigrams(tokens):
    """
    Given a list of tokens like ["machine","learning","approach"],
    returns a list of bigrams like ["machine_learning","learning_approach"].
    """
    return [f"{tokens[i]}_{tokens[i+1]}" for i in range(len(tokens) - 1)]

# Set weighting parameters
W_ACADEMIC_INTEREST = 2.0
W_MAJOR             = 3.0
W_RESEARCH_INTEREST = 2.0
W_UNWEIGHTED        = 1.0  # For the merged, unweighted fields (e.g., extracurriculars, skills, clubs)

# Year-of-study 
BOOST_LOWER_DIVISION = 3.0 # e.g. for freshman/sophomore, < 100 courseID
BOOST_MID_LEVEL      = 2.0  # e.g. for junior/senior, [100..200) courseID
result_b = 1.8
NUM_RECS = 5  # how many recommended courses to show per student

recommendations_data = []  
# For each student, compute combined BM25 scores from their tokenized fields
# Assumes:
bm25 = BM25Okapi(tokenized_courses)
top_scores = []
for i, student_row in df_students.iterrows():
    # Extract each token list from df_students
    academic_tokens  = student_row['tokens_academic_interest']
    major_tokens     = student_row['tokens_major']
    # Bigrams for research tokens
    research_unigrams = student_row.get('tokens_research_interest', [])
    research_bigrams  = create_bigrams(research_unigrams)
    # Combine unigrams + bigrams
    research_tokens   = research_unigrams + research_bigrams
    unweighted_tokens = student_row['tokens_unweighted']

    # Get BM25 scores for each token list
    if academic_tokens:
        scores_academic = bm25.get_scores(academic_tokens)
    else:
        scores_academic = np.zeros(len(df_courses))

    if major_tokens:
        scores_major = bm25.get_scores(major_tokens)
    else:
        scores_major = np.zeros(len(df_courses))

    if research_tokens:
        scores_research = bm25.get_scores(research_tokens)
    else:
        scores_research = np.zeros(len(df_courses))

    if unweighted_tokens:
        scores_unweighted = bm25.get_scores(unweighted_tokens)
    else:
        scores_unweighted = np.zeros(len(df_courses))

    #Combine the scores with weights
    combined_scores = (
          W_ACADEMIC_INTEREST  * scores_academic
        + W_MAJOR              * scores_major
        + W_RESEARCH_INTEREST  * scores_research
        + W_UNWEIGHTED         * scores_unweighted
    )

    # apply year-of-study boosts
    year_of_study = student_row['year_of_study_cleaned']  # e.g., "freshman", "junior", etc.
    sims = list(combined_scores * result_b)  # convert to list for easy iteration

    if year_of_study in ['freshman', 'sophomore']:
        # Boost lower-division courses
        for idx in range(len(sims)):
            course_id = df_courses.loc[idx, 'courseId']
            if parse_last_three_digits(course_id) < 100:
                sims[idx] *= BOOST_LOWER_DIVISION
    elif year_of_study in ['junior', 'senior']:
        # Boost mid-level courses
        for idx in range(len(sims)):
            course_id = df_courses.loc[idx, 'courseId']
            last_three = parse_last_three_digits(course_id)
            if 100 <= last_three < 200:
                sims[idx] *= BOOST_MID_LEVEL

    # Final ranking & output
    sims_array = np.array(sims)
    top_indices = sims_array.argsort()[::-1][:NUM_RECS]
    for idx in top_indices:
        top_scores.append(sims_array[idx])

    row_dict = {
        'StudentIndex': i,
        'Name': student_row.get('Name', 'N/A'),
        'YearOfStudy': student_row.get('YearOfStudy', 'N/A')
    }

    # For each of the top 10 courses, add columns in the same row
    for rank, course_idx in enumerate(top_indices, start=1):
        cid    = df_courses.loc[course_idx, 'courseId']
        ctitle = df_courses.loc[course_idx, 'title']
        score  = sims_array[course_idx]

        # e.g. CourseID1, CourseID2..., CourseTitle1..., BM25Score1...
        row_dict[f'CourseID{rank}']     = cid
        row_dict[f'CourseTitle{rank}']  = ctitle
        row_dict[f'BM25Score{rank}']    = round(score, 4)

    # Append row for this student
    recommendations_data.append(row_dict)

# --- 7) Convert to DataFrame and save to CSV ---
df_recommendations = pd.DataFrame(recommendations_data)
df_recommendations.to_csv("./recommendations.csv", index=False)
print("Recommendations saved to 'recommendations.csv' with all top-5 courses on the same row.")

Recommendations saved to 'recommendations.csv' with all top-5 courses on the same row.


In [7]:
def build_student_profile(student_row):
    return f"""
Name: {student_row.get('Name', '')}
Major: {student_row.get('Major', '')}
Year of Study: {student_row.get('YearOfStudy', '')}
Academic Interests: {student_row.get('AcademicInterest', '')}
Extracurricular Activities: {student_row.get('ExtracurricularActivities', '')}
Skills: {student_row.get('Skills', '')}
Club Memberships: {student_row.get('ClubMemberships', '')}
Research Interests: {student_row.get('ResearchInterests', '')}
Languages: {student_row.get('Languages', '')}
GPA: {student_row.get('GPA', '')}
""".strip()

In [8]:
def build_candidate_courses_text(df_courses, candidate_indices, sims_array):
    blocks = []

    for idx in candidate_indices:
        row = df_courses.loc[idx]
        block = f"""
Course ID: {row.get('courseId', '')}
Title: {row.get('title', '')}
Department: {row.get('dept_code', '')}
Subject Area: {row.get('subject_area', '')}
Level: {row.get('obj_level', '')}
College: {row.get('college', '')}
Units: {row.get('units', '')}
Instruction Type: {row.get('instruction_type1', '')}
Online Course: {row.get('online_course', '')}
Description: {row.get('description', '')}
BM25 Score: {round(float(sims_array[idx]), 4)}
""".strip()
        blocks.append(block)

    return "\n\n".join(blocks)

In [9]:
from typing import List
from pydantic import BaseModel

class CourseRecommendation(BaseModel):
    courseId: str
    final_rank: int
    explanation: str

class RecommendationList(BaseModel):
    recommendations: List[CourseRecommendation]

rerank_agent = Agent(
    name="Course Reranker",
    instructions=(
        "You are a course recommendation assistant. "
        "You will receive a student profile and a list of candidate UCSB courses. "
        "Choose the best 3 courses from the candidate list only. "
        "Rank them from best to worst. "
        "Use the student's major, year of study, academic interests, skills, "
        "extracurricular activities, club memberships, and research interests to tailor your recommendations. "
        "Do not invent courses."
    ),
    model=AGENTS_MODEL,   # important
    model_settings=SETTINGS,
    output_type=RecommendationList,
)

In [10]:
def build_rerank_prompt(student_row, df_courses, candidate_indices, sims_array):
    student_profile = build_student_profile(student_row)
    candidate_courses = build_candidate_courses_text(df_courses, candidate_indices, sims_array)

    return f"""
Student profile:
{student_profile}

Candidate courses:
{candidate_courses}

Task:
Choose the best 5 courses for this student from the candidate list.
Return them ranked from 1 to 5 with a short explanation for each.
""".strip()

In [11]:
async def rerank_courses(student_row, df_courses, candidate_indices, sims_array):
    prompt = build_rerank_prompt(student_row, df_courses, candidate_indices, sims_array)
    result = await Runner.run(rerank_agent, prompt)
    return result.final_output

In [12]:

def normalize_course_id(x):
    """
    Normalize course IDs by:
    - converting to string
    - uppercasing
    - collapsing multiple spaces to one
    - stripping leading/trailing whitespace
    """
    x = str(x).upper().strip()
    x = re.sub(r"\s+", " ", x)
    return x

def merge_llm_results(recommendation_output, df_courses, sims_array):
    llm_rows = []

    normalized_df_ids = df_courses["courseId"].astype(str).apply(normalize_course_id)

    for rec in recommendation_output.recommendations:
        rec_course_id_norm = normalize_course_id(rec.courseId)

        match_idx = normalized_df_ids[normalized_df_ids == rec_course_id_norm].index
        if len(match_idx) == 0:
            print(f"No match found for returned courseId: {rec.courseId}")
            continue

        course_idx = match_idx[0]
        row = df_courses.loc[course_idx]

        llm_rows.append({
            "final_rank": rec.final_rank,
            "courseId": row["courseId"],   # keep original formatting from df_courses
            "title": row["title"],
            "dept_code": row["dept_code"],
            "description": row["description"],
            "bm25_score": round(float(sims_array[course_idx]), 4),
            "explanation": rec.explanation
        })

    if not llm_rows:
        print("No valid LLM recommendations matched course IDs in df_courses.")
        return pd.DataFrame(columns=[
            "final_rank", "courseId", "title", "dept_code",
            "description", "bm25_score", "explanation"
        ])

    return pd.DataFrame(llm_rows).sort_values("final_rank")

In [17]:
# random seed for reproducibility
np.random.seed(369)

# Pick a random student
random_idx = np.random.randint(len(df_students))
student_row = df_students.iloc[random_idx]

print(f"Random student index: {random_idx}")
print(f"Student name: {student_row.get('Name', 'N/A')}")

academic_tokens   = student_row['tokens_academic_interest']
major_tokens      = student_row['tokens_major']
research_unigrams = student_row.get('tokens_research_interest', [])
research_bigrams  = create_bigrams(research_unigrams)
research_tokens   = research_unigrams + research_bigrams
unweighted_tokens = student_row['tokens_unweighted']

scores_academic = bm25.get_scores(academic_tokens) if academic_tokens else np.zeros(len(df_courses))
scores_major = bm25.get_scores(major_tokens) if major_tokens else np.zeros(len(df_courses))
scores_research = bm25.get_scores(research_tokens) if research_tokens else np.zeros(len(df_courses))
scores_unweighted = bm25.get_scores(unweighted_tokens) if unweighted_tokens else np.zeros(len(df_courses))

combined_scores = (
      W_ACADEMIC_INTEREST  * scores_academic
    + W_MAJOR              * scores_major
    + W_RESEARCH_INTEREST  * scores_research
    + W_UNWEIGHTED         * scores_unweighted
)

sims_array = np.array(combined_scores * result_b)
top_indices = sims_array.argsort()[::-1][:10]

recommendation_output = await rerank_courses(student_row, df_courses, top_indices, sims_array)
final_df = merge_llm_results(recommendation_output, df_courses, sims_array)

# ---- student info table ----
student_info = {
    "StudentIndex": random_idx,
    "Name": student_row.get("Name", ""),
    "Major": student_row.get("Major", ""),
    "YearOfStudy": student_row.get("YearOfStudy", ""),
    "AcademicInterest": student_row.get("AcademicInterest", ""),
    "ExtracurricularActivities": student_row.get("ExtracurricularActivities", ""),
    "Skills": student_row.get("Skills", ""),
    "ClubMemberships": student_row.get("ClubMemberships", ""),
    "ResearchInterests": student_row.get("ResearchInterests", ""),
    "Languages": student_row.get("Languages", ""),
    "GPA": student_row.get("GPA", "")
}

student_info_df = pd.DataFrame([student_info])

print("Selected student:")
display(student_info_df)

# ---- add student info to each recommendation row ----
for col, value in student_info.items():
    final_df[col] = value

# reorder columns
final_df = final_df[
    [
        "StudentIndex", "Name", "Major", "YearOfStudy", "AcademicInterest",
        "ExtracurricularActivities", "Skills", "ClubMemberships",
        "ResearchInterests", "Languages", "GPA",
        "final_rank", "courseId", "title", "dept_code",
        "bm25_score", "explanation", "description"
    ]
]

final_df

Random student index: 959
Student name: Student 960
Selected student:


,StudentIndex,Name,Major,YearOfStudy,AcademicInterest,ExtracurricularActivities,Skills,ClubMemberships,ResearchInterests,Languages,GPA
0,959,Student 960,Biology,Senior,Physics,Art Club,Leadership,"Coding Club, Volunteer Group, Music Club, Deba...",Sustainable Agriculture,"German, English, Japanese, Chinese, Spanish",3.31


,StudentIndex,Name,Major,YearOfStudy,AcademicInterest,ExtracurricularActivities,Skills,ClubMemberships,ResearchInterests,Languages,GPA,final_rank,courseId,title,dept_code,bm25_score,explanation,description
0,959,Student 960,Biology,Senior,Physics,Art Club,Leadership,"Coding Club, Volunteer Group, Music Club, Deba...",Sustainable Agriculture,"German, English, Japanese, Chinese, Spanish",3.31,1,ESM 224,SSTNBL WTR RES MGMT,ESM,26.5898,This course on sustainable water resource mana...,Integrates environmental science and managemen...
1,959,Student 960,Biology,Senior,Physics,Art Club,Leadership,"Coding Club, Volunteer Group, Music Club, Deba...",Sustainable Agriculture,"German, English, Japanese, Chinese, Spanish",3.31,2,EEMB 188RE,CONSERV RESTOR SEM,EEMB,24.7586,The seminar on conservation and restoration ec...,Seminar explores current topics in conservatio...
2,959,Student 960,Biology,Senior,Physics,Art Club,Leadership,"Coding Club, Volunteer Group, Music Club, Deba...",Sustainable Agriculture,"German, English, Japanese, Chinese, Spanish",3.31,3,ME 154,STRUCTURES,ME,27.6125,The introductory course in structural analysis...,Introductory course in structural analysis and...


In [18]:
for rec in recommendation_output.recommendations:
    print(rec)

courseId='ESM     224' final_rank=1 explanation="This course on sustainable water resource management aligns perfectly with the student's research interests in sustainable agriculture, providing practical knowledge and skills that can be applied in their field."
courseId='EEMB    188RE' final_rank=2 explanation="The seminar on conservation and restoration ecology is highly relevant to the student's major in Biology and their interest in sustainable practices, offering insights into current issues and hands-on projects."
courseId='ME      154' final_rank=3 explanation="The introductory course in structural analysis and design can enhance the student's understanding of engineering principles, which is beneficial given their interest in physics and leadership skills in project design."


In [15]:
import os
from dotenv import load_dotenv

load_dotenv()
print("OPENROUTER_API_KEY exists:", bool(os.getenv("OPENROUTER_API_KEY")))
print("Key prefix:", os.getenv("OPENROUTER_API_KEY", "")[:12])

OPENROUTER_API_KEY exists: True
Key prefix: sk-or-v1-a08


In [16]:
from openai import OpenAI
import os
from dotenv import load_dotenv

load_dotenv()

client = OpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
)

resp = client.chat.completions.create(
    model="openai/gpt-4o-mini",
    messages=[{"role": "user", "content": "Reply with exactly: hello"}],
)

print(resp.choices[0].message.content)

hello
